# Скок-поскок по "Ситону"

В этой заметке пробегусь по ["основам языка \[Cython\]"](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html). Ситон - это шуточное название языка Сайтон по подобию с Пайтон/Питон, но есть и нешуточная часть: здесь будет теснее и больше к Си, поэтому оставляю статейку по [основам языка Си](https://thecode.media/osnovy-sintaksisa-yazyka-c/).

In [1]:
%load_ext cython

### Знай (не свои) ограничения

Тип `char` - это простой тип, который как будто прост в понимании, правда:

- "char" происходит от "character" - символ, поэтому его можно называть символьным типом;
- правда символ - это целочисленный код, а значит `char` предназначен для хранения целых (integer) чисел;
- размер `char` - это байт, а байт - это минимально адресуемая область памяти, а не всегда 8 бит - значение можно подсмотреть в макросе **CHAR_BIT**.
- также нет ручательства (гарантии), что `char` - это всегда `signed char`

In [2]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

# like `#include <limits.h>`
cimport libc.limits as limits

print(f"CHAR_BIT = {limits.CHAR_BIT}")

cdef signed char c = limits.CHAR_MAX
cdef unsigned char uc = limits.UCHAR_MAX

c += 1   # UB = undefined behaviour
uc += 1  # DB = defined behaviour

print(c, " and ", uc)

CHAR_BIT = 8
-128  and  0


В примере выше удалось:

- [использовать](https://cython.readthedocs.io/en/latest/src/tutorial/external.html) сишные макросы с помощью сайтоновского импорта (cimport)
- посмотреть на переполнение знакового типа, которое считается неопределённым поведением (undefined behaviour)

### Массивы

Массивы - собрания (коолекция) для хранения однородных (гомогенных) элементов. Сайтон поддерживает два стиля объявления [массивов](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#c-arrays): Java и С. Легче и проще пользоваться явовским стилем, потому что он поддерживает объявление и обначаливание (инициализацию) в одну строку.

In [3]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

# like `from libc.math cimport sin`
cdef extern from "math.h":
    double sin(double x)


cdef float[3] point = [0.1, -0.2, 0.3]

sines = [sin(point[i]) for i in range(len(point))]  # Python list

print(sines)

[0.0998334181294999, -0.19866933371588721, 0.29552021804983797]


Примечательно, что вместо сайтоновского импорта `from libc.math cimport sin` можно условно (!) "включить" (include) прототип функции `sin` из заголовочника (заголовочного файла - header file) "math.h", а поскольку это заголовочник из стандартной библиотеки, то местоположение исходника (исходного файла - source file) известно и не нужно прописывать директорию где он расположен. 

### Указатели

Указатели - та тема, на которой Си указал мне на дверь, когда я по-честному пытался его учить. Затем в [Школе 21](https://21-school.ru/), которая тогда ещё была [École 42](https://42.fr/en/homepage/), мне пришлось освоить указатели и тогда я осознал их красоту и мощь. Хотя указатели имеют недобрую и отпугивающую славу, между тем они не столь сложны в освоении, а выгоды их использования колоссальны.

Для начала простой пример и прежде вспомним, что:

- `char` - это целочисленный тип, а символ, например, `'A'` - это код 65 (см. [man ascii](https://man7.org/linux/man-pages/man7/ascii.7.html), а также за [типы](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#types) в Cython)
- немножко за переменные (в условных и грубых выражениях):
  1. пусть была объявлена переменная, например `float x;` -> с этой переменной может быть соотнесена некоторая область памяти
  2. теперь этой переменной присваиваются данные - `x = 21.42` -> данные (21.42) записаны в область памяти, соотнесённую с именем переменной `x`
  3. можно взять местоположение (адрес) вышезаявленной переменной явно с помощью `&x` -> именно адрес переменной `x`, а не данные (21.42) по ней
  4. раз известен адрес переменной `x` как `&x`, то можно перейти по этому адресу напрямую, а значит обратиться к данным с помощью оператора разыменования (dereferncing) `*`, т.е. `*(&x)` (скобки проставил для удобства)

In [4]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stdint cimport uintptr_t

cdef unsigned char uc = 'W'
print(f"uc = {uc} ('{chr(uc)}')")

# ptr is a variable of type `pointer (*) to an unsigned char` 
cdef unsigned char *ptr
# now ptr stores the address (&) of the variable `uc`
ptr = &uc

print(f"address of uc (`&uc`) is {(<uintptr_t>ptr):#x}")

# using `*ptr` leads to error, ptr[0] saves the cell
print(f"value of `*ptr` is {ptr[0]}")

uc = 87 ('W')
address of uc (`&uc`) is 0x782fa9695390
value of `*ptr` is 87


Сайтон оказался не без приколов: вместо взятия данных по адресу из переменной `*ptr` в стиле Си, пришлось писать `ptr[0]`. Сначала мне это не понравилось, ведь ptr - это же не массив, зачем сюда индексация. Но затем я вспомнил об адресной арифметике и всё стало на свои места: `ptr[0] = *(ptr + 0) = *ptr`. Чтобы это прочувствовать, опишу на примере массива, потому что:

- имя массива, пусть `arr`, - это указатель на область памяти, где содержится первый элемент массива
- когда мы хотим взять первый элемент массива, мы обычно пишем `arr[0]`, ведь первый элемент массива - это элемент с нулевым смещением относительно начала массива как ячейки (это тоже область) памяти
- но раз имени `arr` сопоставлен адрес области памяти, значит `arr` очень похож на указатель, ведь эта переменная хранит адрес в памяти, а раз смещение, пусть и нулевое, относительно адреса, то `arr = arr + 0`, а взятие данных из массива происходит через скобки (скобы - они уже квадратные), поэтому `arr[i] = *(arr + i)`

На Си я написал вышний пример как-то так:

```c
#include <stdio.h>
#include <stdint.h>


int main() {
	unsigned char uc = (unsigned char) 'W';  // type casting from `int`
	printf("uc = %hhu (%c)\n", uc, uc);

	unsigned char *ptr = &uc;
	printf("&uc (address of uc) = %p\n", ptr);
	printf("*ptr (fetch from the &uc) = %c\n", *ptr);

	return 0;
}
```

Пока хватит, указатели нужно постигать постепенно, дабы не отпугнуть всем-и-сразу.

### Строки

В Си нет типа данных "строка" (string) как в Python, Java и прочих более высокоуровневых языках (хотя Си тоже высокоуровневый в сравнении с тем же Ассемблером). Итак, строка в Си - это массив символов (array of characters) - пока так и без влияния [Юникода](). Строка как массив символов имеет одну важную особенность: она завершается символом '\0', который называется нуль-терминатор (завершитель) и помечает конец строки.

In [5]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stddef cimport size_t
from libc.stdlib cimport malloc, free
from libc.string cimport strcpy, strcat, strlen

# byte strings are closer to ASCII C strings
cdef const char* s1 = b"Hell\0o"  # not "Hello", but "Hell" because of '\0'
cdef const char* s2 = b"World"   # '\0' is not required by default

cdef size_t siz = strlen(s1) + strlen(s2) + 1  # +1 for '\0' zero-terminator
cdef char* s3 = <char*>malloc(siz)  # allocate `siz * sizeof(char)` bytes and returns pointer to it 

# malloc can fail, therefore it is vital to check the result
if s3 == NULL:  # NULL is not None
    raise MemoryError()

try:
    strcpy(s3, s1)  # writes s1 in the empty buffer s3
    strcat(s3, s2)  # catenates s2 to the end of s1
    print(s3)
finally:
    free(s3)

b'HellWorld'


Разбор примера:

- "сишные" строки - это пайтоновские байтовые строки -> полагаю, что это для обеспечения [ASCII](https://ru.wikipedia.org/wiki/ASCII) режима.
- особое внимание на строку `s1` - завершающий нуль знаменует конец строки и плевать что там идёт дальше после него -> поэтому строка "Hell\0o" в Си будет строкой "Hell", а не "Hello"
- [квалифакторы типов](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#type-qualifiers) поддерживаются
- здесь показано выделение памяти в [куче](https://thecode.media/heap/) с помощью стандартной функции [malloc](https://man7.org/linux/man-pages/man3/malloc.3.html)
- выделенную память нужно осовобождать с помощью [free](https://man7.org/linux/man-pages/man3/free.3p.html) для предотвращения [утечек памяти](https://coddy.tech/docs/ru/c/memory-leaks)

[Здесь](https://cython.readthedocs.io/en/latest/src/tutorial/strings.html) можно почитать за строки в Python, Cython и С - не стоит относиться к строкам небрежно или легкомысленно. Кстати, небольшой пример на связку строк, массивов и указателей и, естественно, адреса на вашей машине (да и на моей тоже) могут разниться.

In [6]:
%%cython -c=-Wall -c=-Werror -c=-Wextra

from libc.stdint cimport uintptr_t

cdef:
    const char* s = b"CAT"
    const char* ptr = s

while (c := ptr[0]) != 0:  # while not the '\0' of the string
    print(f"address={(<uintptr_t>ptr):#x}, char='{chr(c)}'")
    ptr += 1

address=0x782fa9685529, char='C'
address=0x782fa968552a, char='A'
address=0x782fa968552b, char='T'


Можно "обнимать" (embrace) множественные `cdef` утверждения (statements) в [один `cdef:` блок](https://cython.readthedocs.io/en/latest/src/userguide/language_basics.html#grouping-multiple-c-declarations).

Пока хватит.